In [1]:
import pandas as pd
import numpy as np

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/ig-filter-data")

print("Path to dataset files:", path)

100%|██████████| 7.13M/7.13M [00:00<00:00, 44.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/ig-filter-data/versions/1


In [3]:
df = pd.read_csv(path + '/ganjoo_ipl_bbb.csv')

In [4]:
# Create a mapping of unique p_bat to bat names
bat_mapping = df[['p_bat', 'bat']].drop_duplicates().rename(columns={'p_bat': 'player_id', 'bat': 'player_name'})

# Create a mapping of unique p_bowl to bowl names
bowl_mapping = df[['p_bowl', 'bowl']].drop_duplicates().rename(columns={'p_bowl': 'player_id', 'bowl': 'player_name'})

# Combine both mappings into a single DataFrame
player_mapping = pd.concat([bat_mapping, bowl_mapping]).drop_duplicates().reset_index(drop=True)

# Display the first few rows of the mapping
player_mapping.head()

,player_id,player_name
0,5334,Aaron Finch
1,530011,Travis Head
2,5961,Moises Henriques
3,270484,James Faulkner
4,489889,Pat Cummins


players dataset

In [5]:
import pandas as pd

# Extract batter details and rename columns appropriately
df_bat = df[['bat', 'p_bat', 'bat_hand']].rename(
    columns={'bat': 'name', 'p_bat': 'player_id'}
).drop_duplicates(subset=['player_id'])

# Extract bowler details and rename columns appropriately
df_bowl = df[['bowl', 'p_bowl', 'bowl_style']].rename(
    columns={'bowl': 'name', 'p_bowl': 'player_id'}
).drop_duplicates(subset=['player_id'])

# Merge both datasets on 'player_id' using an outer join.
# This ensures that if a player appears as only batter or only bowler, they still show up.
merged = pd.merge(df_bat, df_bowl, on='player_id', how='outer', suffixes=('_bat', '_bowl'))

# Decide which name to keep. If the batter name is available, use it; otherwise use the bowler name.
merged['name'] = merged['name_bat'].combine_first(merged['name_bowl'])
# For bat_hand and bowl_style, they will naturally be NaN if not available, which is acceptable.
merged['bat_hand'] = merged['bat_hand']
merged['bowl_style'] = merged['bowl_style']

# Create the final players_detail DataFrame with the desired columns
players_detail = merged[['name', 'player_id', 'bat_hand', 'bowl_style']]

In [6]:
players_detail

,name,player_id,bat_hand,bowl_style
0,George Bailey,4451,RHB,NaN
1,Dan Christian,4864,RHB,RFM
2,Aaron Finch,5334,RHB,NaN
3,Brad Hogg,5681,LHB,LWS
4,Michael Hussey,5939,LHB,NaN
...,...,...,...,...
466,Raghav Goyal,1350770,NaN,LWS
467,Suyash Sharma,1350792,NaN,LB
468,Shamar Joseph,1356971,NaN,RF
469,Shivam Singh,1380113,RHB,NaN


In [17]:
players_detail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        471 non-null    object
 1   player_id   471 non-null    int64 
 2   bat_hand    425 non-null    object
 3   bowl_style  342 non-null    object
dtypes: int64(1), object(3)
memory usage: 14.8+ KB


In [15]:
dict_bowl_style = {'RF' : ['RF', 'RM', 'RMF', 'RFM'],
 'ROB':['OB', 'OB/LB', 'OB/LBG', 'RM/OB'],
 'LF':['LFM', 'LF', 'LMF', 'LM'],
 'RLB':['LBG', 'LB'],
 'LWS':['LWS'],
 'SLA':['SLA', 'SLA/LWS']}

def update_bowl_style(bowl_style):
    for key, values in dict_bowl_style.items():
        if bowl_style in values:
            return key
    return bowl_style  # Return original value if not found in dictionary

players_detail['bowl_style'] = players_detail['bowl_style'].apply(update_bowl_style)

<ipython-input-15-52795c093336>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  players_detail['bowl_style'] = players_detail['bowl_style'].apply(update_bowl_style)


In [16]:
players_detail['bowl_style'].unique()

array([nan, 'RF', 'LWS', 'LF', 'ROB', 'SLA', 'RLB'], dtype=object)

In [18]:
players_detail.to_csv('ganjoo_ipl_players_detail.csv', index=False)